# ⚖️ Dynamic Legal Knowledge Retrieval for Judgment Prediction (DLKR-LJP)
### Inference-Time Retrieval + Dual-Metric Relevance Gating + `InLegalBERT` Prefix Fusion

**Abstract Implementation:**
- **Dynamic Retrieval Module (DRM):** Dynamically retrieves relevant statutory articles (IPC/BNS), essential legal ingredients, and landmark prior case precedents ahead of fusion.
- **Relevance and Confidence Gate (RCG):** Filters candidates using a dual-metric gate (semantic cosine threshold $\tau \ge 0.65$ + element consistency check) to eliminate noise.
- **Prefix Attention Fusion:** Projects gated legal knowledge into prefix tokens conditioning `law-ai/InLegalBERT` self-attention.
- **3-Head Multi-Task Prediction:**
  1. Law Articles Prediction (IPC 1860 & BNS 2023)
  2. Criminal Charges Classification
  3. Term of Penalty & Statutory Sentencing Severity Tiers

## 1. Verify GPU Hardware (T4 GPU / 15 GB VRAM)

In [ ]:
!nvidia-smi

Wed Oct  7 16:49:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/DLKR_LJP_Indian_Law', exist_ok=True)
os.makedirs('/content/dlkr_ljp_runtime', exist_ok=True)
print("✅ Runtime directories initialized.")

Mounted at /content/drive
✅ Runtime directories initialized.


## 3. Install Dependencies

In [ ]:
!pip install -q transformers>=4.40.0 accelerate>=0.28.0 chromadb sentence-transformers gradio pydantic tabulate scikit-learn
print("✅ Dependencies successfully installed.")

✅ Dependencies successfully installed.


## 3.1 Unpack Project Codebase
Drag and drop `dynamic_legal_retrieval_ljp.zip` into the left Files tab in Colab or put it in Google Drive.

In [ ]:
import os, zipfile

if not os.path.exists('/content/pipeline.py'):
    candidates = [
        '/content/dynamic_legal_retrieval_ljp.zip',
        '/content/drive/MyDrive/dynamic_legal_retrieval_ljp.zip',
        '/content/drive/MyDrive/DLKR_LJP_Indian_Law/dynamic_legal_retrieval_ljp.zip'
    ]
    found_zip = next((c for c in candidates if os.path.exists(c)), None)
    if found_zip:
        print(f'📦 Unpacking project from {found_zip}...')
        with zipfile.ZipFile(found_zip, 'r') as z:
            z.extractall('/content')
        print('✅ Codebase extracted successfully!')
    else:
        print('💡 Tip: Drag and drop dynamic_legal_retrieval_ljp.zip into Colab Files tab, then re-run this cell.')
else:
    print('✅ Project codebase already present in /content.')

📦 Unpacking project from /content/drive/MyDrive/dynamic_legal_retrieval_ljp.zip...
✅ Codebase extracted successfully!


## 4. Initialize DLKR-LJP Pipeline

In [ ]:
import os, sys, glob, zipfile

# 1. Specifically look for the updated dynamic_legal_retrieval_ljp.zip (ignoring old backups)
zip_candidates = (
    glob.glob('/content/dynamic_legal_retrieval_ljp*.zip') +
    glob.glob('/content/drive/MyDrive/**/dynamic_legal_retrieval_ljp*.zip', recursive=True) +
    glob.glob('/content/*.zip')
)
valid_zips = [z for z in zip_candidates if 'muhsina_judgement_prediction' not in os.path.basename(z).lower()]

if valid_zips:
    chosen_zip = valid_zips[0]
    print(f'📦 Unpacking updated project zip: {chosen_zip} into /content...')
    with zipfile.ZipFile(chosen_zip, 'r') as zf:
        zf.extractall('/content')
    print('✅ Updated codebase successfully extracted!')
else:
    print('⚠️ Please upload dynamic_legal_retrieval_ljp.zip to Colab Files tab on the left (📁 icon).')

# 2. Add /content to sys.path
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# 3. Import Pipeline dynamically with fallback across all class names
import pipeline
PipelineClass = getattr(pipeline, 'DynamicLjpPipeline', None) or \
                getattr(pipeline, 'DynamicLegalRetrievalLjpPipeline', None) or \
                getattr(pipeline, 'DlkrLjpPipeline', None)

if PipelineClass is None:
    import inspect
    avail = [name for name, cls in inspect.getmembers(pipeline, inspect.isclass)]
    raise ImportError(f'❌ Outdated pipeline.py found with classes {avail}. Please upload the new dynamic_legal_retrieval_ljp.zip!')

pipeline = PipelineClass(load_transformer=True)
print(f'⚖️ DLKR-LJP Pipeline successfully initialized using {PipelineClass.__name__}!')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  534MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[DLKR-LJP] Successfully loaded law-ai/InLegalBERT.
✅ DLKR-LJP Pipeline initialized with InLegalBERT backbone.


## 5. Run Sample Case Adjudication (Cheating & Forgery)

In [ ]:
sample_case = """
The accused persons dishonestly induced the complainant to transfer Rs. 85 Lakhs against promises of allotment of commercial land.
Investigation revealed that title deeds, town planning certificates, and official stamps produced by the accused were forged.
Cheques issued by the accused subsequently bounced due to closed accounts.
"""

result = pipeline.run(sample_case)

print("\n--- 1. RELEVANCE & CONFIDENCE GATE AUDIT ---")
print(f"Passed Gate Items (Fused into Prefix): {len(result['gated_knowledge'])}")
for item in result['gated_knowledge']:
    print(f"  [PASSED] {item['title']} (Similarity: {item['similarity']:.2f})")
print(f"Filtered Noise Items (Discarded): {len(result['filtered_noise'])}")
for item in result['filtered_noise']:
    print(f"  [REJECTED] {item['title']}: {item['reason']}")

print("\n--- 2. 3-HEAD PREDICTIONS ---")
print("Head 1 (Articles):", [(a['display_name'], f"{a['probability']:.1%}") for a in result['predictions']['head1_articles'][:3]])
print("Head 2 (Charges):", [(c['charge_name'], f"{c['probability']:.1%}") for c in result['predictions']['head2_charges'][:2]])
print("Head 3 (Penalty):", result['predictions']['primary_penalty']['tier'])

print("\n--- 3. TRACEABLE EVIDENCE ---")
print(result['traceable_evidence'])


[DLKR Pipeline] === Processing Legal Case Fact Narrative ===
[Step 1/4] Dynamically retrieving candidate articles, elements, and prior cases from external KB...
         Retrieved 6 candidate knowledge items.
[Step 2/4] Applying Relevance and Confidence Gate (Similarity Cutoff & Element Consistency)...
         Gate Decision: 0 PASSED | 6 DISCARDED AS NOISE.
[Step 3/4] Constructing gated prefix-attention token representations...
[Step 4/4] Executing 3-Head Multi-Task Prediction (Articles, Charges, Penalty)...


model.safetensors: reconstructing file:   0%|          |  0.00B /  534MB            

model.safetensors: downloading bytes:           |  0.00B            

[DLKR Pipeline] Adjudication complete.


--- 1. RELEVANCE & CONFIDENCE GATE AUDIT ---
Passed Gate Items (Fused into Prefix): 0
Filtered Noise Items (Discarded): 6
  [REJECTED] Cheating and Dishonestly Inducing Delivery (IPC Section 420 / BNS Section 318(4)): Low Confidence Gate: Similarity 0.529 is below threshold tau (0.55)
  [REJECTED] State of Haryana v. Bhajan Lal (1992) Supp (1) SCC 335: Low Confidence Gate: Similarity 0.491 is below threshold tau (0.55)
  [REJECTED] Criminal Breach of Trust (IPC Section 406 / BNS Section 316): Low Confidence Gate: Similarity 0.430 is below threshold tau (0.55)
  [REJECTED] Using as Genuine a Forged Document (IPC Section 471 / BNS Section 340(2)): Low Confidence Gate: Similarity 0.426 is below threshold tau (0.55)
  [REJECTED] Forgery for Purpose of Cheating (IPC Section 468 / BNS Section 336(3)): Low Confidence Gate: Similarity 0.419 is below threshold tau (0.55)
  [REJECTED] Forgery of Valuable Security or Will (IPC Section 467 / BNS Section 338

## 6. Launch Interactive Gradio Dashboard (Public URL)

In [ ]:
from app import demo
# share=True creates a public https://xxxx.gradio.live URL!
demo.launch(share=True)

/content/app.py:93: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="DLKR-LJP: Dynamic Legal Knowledge Retrieval", theme=theme) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6466fdb2770fddff56.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7. Run 8-Domain Accuracy & Judicial Consistency Benchmark
Validates all 20 statutory IPC/BNS articles across all legal gradients (Homicide, Attempt, Hurt, Theft, Robbery, Breach of Trust, Cheating, Forgery, Dowry, Cruelty, Intimidation, Rape, and Civil Quashing).

In [ ]:
import os, sys, glob, zipfile

# 1. Clear out-of-date cached modules from Python memory
for mod in list(sys.modules.keys()):
    if mod.startswith(('pipeline', 'gating', 'models', 'retrieval', 'config', 'tests')):
        del sys.modules[mod]

# 2. Locate and unpack the new dynamic_legal_retrieval_ljp.zip
zip_candidates = [
    '/content/dynamic_legal_retrieval_ljp.zip',
    '/content/drive/MyDrive/dynamic_legal_retrieval_ljp.zip'
] + glob.glob('/content/*dynamic_legal_retrieval_ljp*.zip') + glob.glob('/content/drive/MyDrive/**/*dynamic_legal_retrieval_ljp*.zip', recursive=True)

target_zip = next((z for z in zip_candidates if os.path.exists(z)), None)
if not target_zip:
    raise FileNotFoundError(
        '❌ dynamic_legal_retrieval_ljp.zip was NOT found in /content!\n'
        '👉 Please drag and drop dynamic_legal_retrieval_ljp.zip into the Colab Files tab on the left (📁 icon).'
    )

print(f'📦 Extracting updated codebase from: {target_zip}...')
with zipfile.ZipFile(target_zip, 'r') as zf:
    zf.extractall('/content')
print('✅ Codebase extracted successfully!')

os.chdir('/content')
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# 3. Run complete 8-Domain Accuracy Benchmark Suite
print('\n=== RUNNING 8-DOMAIN BENCHMARK SUITE ===')
!python -m unittest tests/test_all_sections_accuracy.py

# 4. Initialize 5-Step DLKR-LJP Pipeline with InLegalBERT & JCV
from pipeline import DynamicLjpPipeline
pipeline = DynamicLjpPipeline(load_transformer=True)
print('\n⚖️ DLKR-LJP 5-Step Pipeline initialized successfully!')

# 5. Interactive Multi-Domain Showcase
demo_cases = [
    ('Sudden Fight / Culpable Homicide (Exception 4)', 'A contractor argued with a labourer over unpaid wages. The labourer shoved him, and the contractor picked up an iron rod lying nearby and struck once on the head. Labourer died next morning. No prior animosity or planning.'),
    ('Robbery at Knife-Point (IPC 392)', 'The accused accosted the commuter on an isolated street, brandished a large knife at his throat, and forcibly extorted his wallet, cash, and smartphone under threat of instant death.'),
    ('Dangerous Weapon Hurt (IPC 324)', 'The accused attacked the victim with a sharp knife causing a deep bleeding laceration on the forearm. The victim was treated with stitches as injuries were non-fatal and non-grievous.'),
    ('Dowry Death (IPC 304B)', 'Within three years of marriage, the young bride died of severe burn injuries under abnormal circumstances at her matrimonial home soon after persistent harassment and torture for dowry.'),
    ('Civil Quashing u/s 482 (Bhajan Lal)', 'Dispute over unpaid consulting invoices and delay in delivery of software source code. Dispute was purely civil and breach of contract only with no criminal offence made out.')
]

for label, narrative in demo_cases:
    print(f'\n================== {label} ==================')
    out = pipeline.run(narrative)
    p = out['predictions']
    print('Top Article:    ', p['head1_articles'][0]['display_name'], f"({p['head1_articles'][0]['probability']*100:.1f}%)")
    print('Top Charge:     ', p['head2_charges'][0]['charge_name'], f"({p['head2_charges'][0]['probability']*100:.1f}%)")
    print('Primary Penalty:', p['primary_penalty']['tier'])
    if 'judicial_consistency_audit' in p:
        audit = p['judicial_consistency_audit']
        print('JCV Audit:      ', audit['audit_status'], f"(JCI: {audit['judicial_consistency_index']}%)")
